### Installing required packages    

In [ ]:
!pip install google-cloud-storage
!pip install --upgrade google-cloud-storage


### Google cloud 

In [3]:
from google.cloud import storage


In [2]:
try:
    # If you need to specify a project explicitly:
    # storage_client = storage.Client(project=PROJECT_ID)
    storage_client = storage.Client()
    print("Google Cloud Storage client initialized successfully.")
except Exception as e:
    print(f"Error initializing GCS client: {e}")
    print("Please ensure your authentication is set up correctly (e.g., `gcloud auth application-default login` or Colab authentication).")
    exit() # Exit if client cannot be initialized

# --- 1. List all buckets in your project ---
print("\n--- Listing all buckets in your project ---")
try:
    buckets = storage_client.list_buckets()
    for bucket in buckets:
        print(f"Bucket Name: {bucket.name}")
except Exception as e:
    print(f"Error listing buckets: {e}")

Google Cloud Storage client initialized successfully.

--- Listing all buckets in your project ---
Bucket Name: apps-cityvision-prod_cloudbuild
Bucket Name: archive-cityvision
Bucket Name: dataflow-cityvision
Bucket Name: gcf-v2-sources-220669230854-us-west1
Bucket Name: gcf-v2-uploads-220669230854.us-west1.cloudfunctions.appspot.com
Bucket Name: processed-cityvision
Bucket Name: raw-cityvision
Bucket Name: run-sources-apps-cityvision-prod-us-west1


In [ ]:
storage_client = storage.Client()
bucket = storage_client.bucket("processed-cityvision")
blobs = bucket.list_blobs()
print(f"the blobs in the bucket {bucket} are :")
for blob in blobs:
    print(blob.name)
    break

the blobs in the bucket <Bucket: processed-cityvision> are :
00f93ddd-0c26-5ff2-b05e-175c651bf86a/1692_SCU2VN_2024-10-01_2345/Standard_SCU2VN_2024-10-01_2345.001.cut


### YOLO Training Code

In [ ]:
!pip install ultralytics

^C


   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 6.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/39.0 MB ? eta -:--:--
   -------- ------------------------------- 8.1/39.0 MB 42.0 MB/s eta 0:00:01
   ----------------- ---------------------- 17.0/39.0 MB 42.9 MB/s eta 0:00:01
   -------------------------- ------------- 26.0/39.0 MB 43.0 MB/s eta 0:00:01
   -------------------------------- ------- 31.7/39.0 MB 38.9 MB/s eta 0:00:01
   -------------------------------------- - 37.5/39.0 MB 36.6 MB/s eta 0:00:01
   ---------------------------------------  38.8/39.0 MB 35.4 MB/s eta 0:00:01
   ---------------------------------------- 39.0/39.0 MB 31.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------- ----------------- 7.1/12.6 MB 36.5 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 33.9 MB/s eta 0:00:00
   ---

In [23]:
# Import necessary libraries
import os
from google.cloud import storage
from ultralytics import YOLO
import yaml
import shutil
import random
import cv2
import numpy as np


Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\MJATTU\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [28]:

# --- Configuration ---
GCS_BUCKET_NAME = "open-cityvision"
GCS_DATA_PATH = "Dataset/" # Path *within* your GCS bucket to the dataset root
LOCAL_DATA_DIR = "yolo_dataset/" # Local directory to download data to


In [29]:
# YOLO Model Configuration
MODEL_VERSION = "yolov11n.pt" 
IMG_SIZE = 640
BATCH_SIZE = 16
DEVICE = 0 # 0 for GPU (if available), 'cpu' for CPU
EPOCHS = 10

In [32]:
# --- 1. Function to Download Data from GCS ---
def download_data_from_gcs(bucket_name, gcs_path, local_dir):
    """
    Downloads a directory and its contents from a Google Cloud Storage bucket.

    Args:
        bucket_name (str): The name of the GCS bucket.
        gcs_path (str): The path to the directory in GCS (e.g., "datasets/my_data/").
        local_dir (str): The local directory to save the downloaded files.
    """
    print(f"Attempting to download data from gs://{bucket_name}/{gcs_path} to {local_dir}")
    
    storage_client = storage.Client()
    try:
        storage_client = storage.Client()
        print("the stroage_client passed")
        bucket = storage_client.bucket(bucket_name)
        print("the bucket passed, bucket_name is ", bucket_name)

        # Ensure local directory exists
        os.makedirs(local_dir, exist_ok=True)

        blobs = bucket.list_blobs(prefix=gcs_path) # List all blobs with the given prefix
        print( "the gcs path is ", gcs_path)
        print("the blobs are", blobs)
        downloaded_count = 0
        for blob in blobs:
            # Construct local file path, preserving directory structure
            relative_path = os.path.relpath(blob.name, gcs_path)
            print(" the relative path is ", relative_path)
            local_file_path = os.path.join(local_dir, relative_path)

            # Create subdirectories if they don't exist
            os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

            if not blob.name.endswith('/'): # Skip directories themselves
                blob.download_to_filename(local_file_path)
                downloaded_count += 1
                print(f"Downloaded: {blob.name} -> {local_file_path}")

        if downloaded_count == 0:
            print(f"No files found or downloaded from gs://{bucket_name}/{gcs_path}. "
                  f"Please check bucket name and GCS path.")
        else:
            print(f"Successfully downloaded {downloaded_count} files from GCS.")

    except Exception as e:
        print("the exception is ", e)
        print(f"Error downloading data from GCS: {e}")
        print("Please ensure your Google Cloud credentials are set up correctly.")
        print("You can use `gcloud auth application-default login` or set the "
              "`GOOGLE_APPLICATION_CREDENTIALS` environment variable.")
        exit(1) # Exit if data download fails



In [33]:
download_data_from_gcs(GCS_BUCKET_NAME, GCS_BUCKET_NAME, LOCAL_DATA_DIR)

Attempting to download data from gs://open-cityvision/open-cityvision to yolo_dataset/
the stroage_client passed
the bucket passed, bucket_name is  open-cityvision
the gcs path is  open-cityvision
the blobs are <google.api_core.page_iterator.HTTPIterator object at 0x00000137111F6710>
No files found or downloaded from gs://open-cityvision/open-cityvision. Please check bucket name and GCS path.


In [ ]:
# --- 2. Data Augmentation (Example using Albumentations for custom augmentation) ---
# Ultralytics YOLO models have built-in augmentation. This section shows how you could
# add *additional* custom augmentation if needed, or modify the dataset directly.
# For simplicity and leveraging Ultralytics' robust pipeline, we'll primarily rely
# on its built-in augmentation, but this demonstrates the concept.

# If you need more control, you'd typically create a custom PyTorch Dataset
# and apply augmentations there. Ultralytics handles much of this internally.

# Example of a simple custom augmentation function (not directly integrated into YOLO training here)
def custom_augment_image_and_labels(image_path, label_path):
    """
    Applies a simple custom augmentation (e.g., random flip) to an image and its YOLO labels.
    This is illustrative; Ultralytics handles augmentation internally.
    """
    image = cv2.imread(image_path)
    if image is None:
        print(f"Warning: Could not load image {image_path}")
        return None, None

    height, width, _ = image.shape
    labels = []
    try:
        with open(label_path, 'r') as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                labels.append(parts)
    except FileNotFoundError:
        print(f"Warning: Label file not found for {image_path}")
        return None, None

    # Example: Random horizontal flip
    if random.random() > 0.5:
        image = cv2.flip(image, 1) # Flip horizontally
        for label in labels:
            label[1] = 1 - label[1] # Adjust x_center for flip

    # You could add more augmentations here (e.g., rotation, scale, color jitter)
    # For more complex augmentations with bounding box support, consider Albumentations.

    # This function would then save the augmented image/labels to a temporary folder
    # or be part of a custom PyTorch Dataset.
    # For Ultralytics, built-in augmentations are usually sufficient and more efficient.
    return image, labels



In [ ]:
# --- 3. Function to Train YOLO Model ---
def train_yolo_model(data_yaml_path, model_version, epochs, img_size, batch_size, device):
    """
    Trains a YOLO model using the Ultralytics framework.

    Args:
        data_yaml_path (str): Path to the dataset configuration YAML file.
        model_version (str): The pre-trained YOLO model to use (e.g., 'yolov8n.pt').
        epochs (int): Number of training epochs.
        img_size (int): Image size for training.
        batch_size (int): Batch size for training.
        device (int/str): Device to train on (e.g., 0 for GPU, 'cpu').
    """
    print(f"\n--- Starting YOLO Model Training with {model_version} ---")
    try:
        # Load a pre-trained YOLO model
        # For YOLOv11, if it's released and compatible, you'd load it here.
        model = YOLO(model_version)

        # Train the model
        # Ultralytics' 'train' method automatically applies robust data augmentation
        # (e.g., mosaic, copy-paste, random flip, scale, translate, perspective, hue, saturation, value).
        # You can customize these in the data.yaml or by passing specific arguments to .train()
        # For more details on augmentation, refer to Ultralytics documentation:
        # https://docs.ultralytics.com/usage/cfg/#augment-settings
        results = model.train(
            data=data_yaml_path,
            epochs=epochs,
            imgsz=img_size,
            batch=batch_size,
            device=device,
            # Additional augmentation parameters can be set here or in data.yaml
            # e.g., degrees=10, translate=0.1, scale=0.5, shear=0.1, perspective=0.0002,
            # flipud=0.5, fliplr=0.5, mosaic=1.0, mixup=0.0, copy_paste=0.0,
            # auto_augment="randaugment" # for advanced auto-augmentation policies
            # You can also disable default augmentations if you implement your own fully
            # augment=False # Use with caution, as it will disable all default augmentations
        )

        print("\n--- Training Complete! ---")
        print(f"Results saved to: {model.trainer.save_dir}")
        print("You can find the best.pt and last.pt models in this directory.")

    except Exception as e:
        print(f"Error during YOLO model training: {e}")
        print("Please ensure Ultralytics is installed (`pip install ultralytics`) "
              "and your dataset `data.yaml` is correctly configured.")
        exit(1)



In [ ]:
# --- Main Execution Flow ---
if __name__ == "__main__":
    # 1. Download data from GCS
    download_data_from_gcs(GCS_BUCKET_NAME, GCS_DATA_PATH, LOCAL_DATA_DIR)

    # 2. Verify data.yaml path
    # Example: If your GCS_DATA_PATH contains 'data.yaml' directly, it will be at LOCAL_DATA_DIR/data.yaml
    DATA_YAML_FILE = os.path.join(LOCAL_DATA_DIR, "data.yaml")

    if not os.path.exists(DATA_YAML_FILE):
        print(f"Error: {DATA_YAML_FILE} not found. Please ensure your GCS_DATA_PATH "
              f"contains a data.yaml file or specify its correct location.")
        exit(1)

    # Verify data.yaml content 
    try:
        with open(DATA_YAML_FILE, 'r') as f:
            data_config = yaml.safe_load(f)
            print("\n--- Loaded data.yaml configuration ---")
            print(yaml.dump(data_config, indent=2))
            # Ensure paths in data.yaml are relative to the LOCAL_DATA_DIR
            # or are absolute paths to the downloaded data.
            # Example: train: images/train/
            #          val: images/val/
            #          names: [...]
    except Exception as e:
        print(f"Error loading data.yaml: {e}")
        exit(1)

    # 3. Train the YOLO model
    train_yolo_model(
        data_yaml_path=DATA_YAML_FILE,
        model_version=MODEL_VERSION,
        epochs=EPOCHS,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        device=DEVICE
    )

    # --- Clean up ---
    print(f"\nCleaning up: Removing local data directory {LOCAL_DATA_DIR}")
    shutil.rmtree(LOCAL_DATA_DIR)
